In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

df = pd.read_csv("telecom_churn.csv")
print(f"Loaded: {df.shape[0]} Customers, {df.shape[1]} Columns")
print(f"Churn Rate: {df['churn'].mean():.1%}")
df.head()

Loaded: 2000 Customers, 10 Columns
Churn Rate: 22.9%


,customer_id,tenure_months,age,monthly_charges,total_charges,contract_type,payment_method,region,signup_date,churn
0,CUST00001,22,40,79.58,1751.54,One year,Mailed check,West,2022-08-03,1
1,CUST00002,72,30,96.73,7345.50,Month-to-month,Credit card,East,2022-08-08,1
2,CUST00003,60,43,87.25,5078.43,Two year,Electronic check,North,2023-03-17,0
3,CUST00004,20,25,97.50,1924.71,One year,Mailed check,West,2021-10-12,0
4,CUST00005,1,27,62.24,62.28,Month-to-month,Credit card,North,2023-06-09,1


In [2]:
print(df.dtypes)

customer_id            str
tenure_months        int64
age                  int64
monthly_charges    float64
total_charges      float64
contract_type          str
payment_method         str
region                 str
signup_date            str
churn                int64
dtype: object


In [3]:
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X train Shape:\n", X_train.shape)
print("\nX test Shape:\n", X_test.shape)

X train Shape:
 (1600, 8)

X test Shape:
 (400, 8)


Part A — encoding and scaling categorical & numeric columns  
TASK 01  
One-hot, ordinal, and target encoding side by side

In [4]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


# Saperate Columns
num_col = ['tenure_months', 'age', 'monthly_charges', 'total_charges']
cat_col = ['contract_type', 'payment_method', 'region']
date_col = ['signup_date']
id_col = ['customer_id']


print('Numeric Columns:', num_col)
print('Categorical Columns:', cat_col)
print("Date Columns:", date_col)
print('ID Column:', id_col)

Numeric Columns: ['tenure_months', 'age', 'monthly_charges', 'total_charges']
Categorical Columns: ['contract_type', 'payment_method', 'region']
Date Columns: ['signup_date']
ID Column: ['customer_id']


In [5]:
# Saperate Numeric and Categorical Data
X_cat_train = X_train[cat_col]
X_cat_test = X_test[cat_col]

X_num_train = X_train[num_col].fillna(0)
X_num_test = X_test[num_col].fillna(0)

# Scale numeric Feature
scaler = StandardScaler()
X_num_train_s = scaler.fit_transform(X_num_train)
X_num_test_s = scaler.transform(X_num_test)

In [6]:
# One HOt encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_ohe = ohe.fit_transform(X_train[cat_col])

# X_train_ohe = np.hstack([X_num_train, X_train_ohe])

model_ohe = LogisticRegression(max_iter=1000)

score_ohe = cross_val_score(
    model_ohe, X_train_ohe, y_train,
    cv = 5, scoring = "roc_auc"
).mean()

print(f"One hot encoding CV AUC : {score_ohe:.4f}")

One hot encoding CV AUC : 0.6987


* One-hot: Best for nominal categories because it does not assume any order between categories.

In [7]:
# Ordinal Encoding
oe = OrdinalEncoder()
X_train_oe = oe.fit_transform(
    X_train[cat_col]
)

X_train_oe = np.hstack([
    X_num_train, X_train_oe
])

model_oe = LogisticRegression(max_iter=1000)

score_oe = cross_val_score(
    model_oe, X_train_oe, y_train,
    cv = 5, scoring = 'roc_auc'
).mean()

print(f"Ordinal Encoding : {score_oe:.4f}")

Ordinal Encoding : 0.7419


* Ordinal: Best when categories have a natural order, such as contract length.

In [8]:
# Target Encoding
X_train_target = X_train[cat_col].copy()

for col in cat_col:
    means = X_train.assign(
        churn=y_train.values
    ).groupby(col)['churn'].mean()

    X_train_target[col] = X_train[col].map(means)

# Combine numeric + Target Encoding categorical
X_train_target = np.hstack([
    X_num_train, X_train_target.values
])

model_target = LogisticRegression(max_iter=1000)

score_target = cross_val_score(
    model_target, X_train_target, y_train,
    cv = 5, scoring = 'roc_auc'
).mean()

print(f"Target Mean CV AUC : {score_target:.4f}")

Target Mean CV AUC : 0.7506


* Target encoding: Useful for categorical variables when their relationship with the target is informative, because each category is replaced by its mean churn rate.

In [9]:
# Compare the three CV Scores:-

results = pd.DataFrame({
    "Encoding": [
        "One Hot",
        "Ordinal",
        "Target Mean"
    ],
    "CV Scores": [
        score_ohe.mean(),
        score_oe.mean(),
        score_target.mean()
    ]
})

print(results)

      Encoding  CV Scores
0      One Hot   0.698686
1      Ordinal   0.741946
2  Target Mean   0.750649


* The scores differ because each encoding represents the categorical information differently, which changes what the Logistic Regression model can learn.

TASK 02  
Scaling: standard, min-max, and robust  
Show that the scalers differ, and know which one your data needs.

In [10]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

num_col = ['tenure_months', 'age', 'monthly_charges', 'total_charges']

# Scalers
scalers = {
    "Standard Scaler" : StandardScaler(),
    "MinMax Scaler"   : MinMaxScaler(),
    "Robust Scaler"   : RobustScaler()
}

# Compare cross validation score
for name, scaler in scalers.items():
    model = make_pipeline(
        scaler, LogisticRegression(max_iter=1000)
    )

    scores = cross_val_score(
        model, X_num_train, y_train, cv=5
    )

    print(name, "CV Score:", scores.mean())

Standard Scaler CV Score: 0.765625
MinMax Scaler CV Score: 0.7675000000000001
Robust Scaler CV Score: 0.765625


In [11]:
# Check outliers in monthly charges and total charges:
print("Monthly Charges")
print(X_train['monthly_charges'].describe())
print("\nTotal Charges")
print(X_train['total_charges'].describe())

Monthly Charges
count    1600.000000
mean       64.373156
std        24.724243
min        18.000000
25%        47.307500
50%        63.455000
75%        81.140000
max       150.000000
Name: monthly_charges, dtype: float64

Total Charges
count    1564.000000
mean     1417.005320
std      1508.468502
min        17.730000
25%       312.887500
50%       891.635000
75%      1952.377500
max      8337.180000
Name: total_charges, dtype: float64


In [12]:
# To check the largest value
print("Largest Monthly Charges")
print(X_train['monthly_charges'].nlargest(10))
print("Largest Total Charges")
print(X_train['total_charges'].nlargest(10))

Largest Monthly Charges
153     150.00
355     145.84
1739    143.15
1204    141.13
468     138.98
779     137.47
842     135.65
111     133.20
658     132.55
1297    132.39
Name: monthly_charges, dtype: float64
Largest Total Charges
1358    8337.18
280     8261.89
1001    8103.53
1604    7776.43
1454    7692.70
1204    7690.39
468     7363.53
1       7345.50
56      7313.88
1253    7054.73
Name: total_charges, dtype: float64


* Observation: Very high values in monthly_charges and especially total_charges can look like outliers. RobustScaler is designed to handle outliers better because it uses the median and IQR instead of the mean and standard deviation.
* RobustScaler beats StandardScaler and MinMaxScaler when the numeric features contain significant outliers because it is less affected by extreme values.

Part B — creating features: polynomial, interaction, and dates  
TASK 03  
Polynomial and interaction features that earn their place  
The curriculum's polynomial-features activity, judged with numbers, not vibes.

In [13]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression, Ridge

# Select a sensible subset of numeric features
num_set = ["monthly_charges", "total_charges"]
X_train_numeric = X_train[num_set].copy()
X_train_numeric = X_train_numeric.fillna(0)

# Add polynomial features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train_numeric)

# Compare CV score without and with polynomial features
model = LogisticRegression(max_iter=1000)

CV_numeric = cross_val_score(
    model, X_train_numeric, y_train,
    cv=5, scoring='accuracy'
).mean()

CV_poly = cross_val_score(
    model, X_poly, y_train,
    cv=5, scoring='accuracy'
).mean()

comparison = pd.DataFrame({
    "Features" : ["Original", "Polynomial Degree 2"],
    "CV Scores" : [CV_numeric, CV_poly]
})

print(comparison)

              Features  CV Scores
0             Original      0.765
1  Polynomial Degree 2      0.760


In [14]:
# Calculate Improvement
improvement = CV_poly - CV_numeric

print(f"Original CV Score: {CV_numeric:.4f}")
print(f"Polynomial CV Score: {CV_poly:.4f}")
print(f"Improvement: {improvement:.4f}")


if improvement > 0:
    print("\nPolynomial features helped.")
else:
    print("\nPolynomial did not helped.")

Original CV Score: 0.7650
Polynomial CV Score: 0.7600
Improvement: -0.0050

Polynomial did not helped.


In [15]:
# show polynomial features name
feature_name = poly.get_feature_names_out(num_set)
print("Polynomial Features:")
print(feature_name)

Polynomial Features:
['monthly_charges' 'total_charges' 'monthly_charges^2'
 'monthly_charges total_charges' 'total_charges^2']


In [16]:
# Fit polynomial model and check coefficients
model.fit(X_poly, y_train)

poly_coefficients = pd.DataFrame({
    "Features" : feature_name,
    "Coefficient" : model.coef_[0]
})

print("Polynomial Features Coefficients:")
print(poly_coefficients)

Polynomial Features Coefficients:
                        Features   Coefficient
0                monthly_charges -3.632000e-02
1                  total_charges -5.362736e-04
2              monthly_charges^2  3.008283e-04
3  monthly_charges total_charges  6.616226e-06
4                total_charges^2 -5.205835e-08


In [17]:
# Identify the strongest interaction
interaction_rows = poly_coefficients[
    poly_coefficients["Features"].str.contains("")
]

if len(interaction_rows) > 0:
    strongest_interaction = interaction_rows.loc[
        interaction_rows["Coefficient"].abs().idxmax()
    ]
    print("Strongest Interaction")
    print(strongest_interaction)

Strongest Interaction
Features       monthly_charges
Coefficient           -0.03632
Name: 0, dtype: object


In [18]:
# Rerun the better version under Ridge
ridge = Ridge(alpha=1.0)

ridge_original = cross_val_score(
    ridge, X_train_numeric, y_train, cv=5, scoring="r2"
).mean()

ridge_poly = cross_val_score(
    ridge, X_poly, y_train, cv=5, scoring="r2"
).mean()

ridge_comparison = pd.DataFrame({
    "Features" : ["Original", "Polynomial Degree 2"],
    "Ridge CV Score" : [ridge_original, ridge_poly]
})

print("Ridge Comparison:")
print(ridge_comparison)

Ridge Comparison:
              Features  Ridge CV Score
0             Original        0.044386
1  Polynomial Degree 2        0.045732


In [19]:
# Check whether Ridge helped control the extra terms
ridge.fit(X_poly, y_train)

ridge_coefficients = pd.DataFrame({
    "Features" : feature_name,
    "Coefficient" : ridge.coef_
})

print("Ridge Polynomial Coefficients:")
print(ridge_coefficients)

# Maximum absolute coefficient
max_coef = np.max(np.abs(ridge.coef_))
print("\nLargest absolute Ridge coefficient:", max_coef)

if max_coef < 10:
    print("\nThe extra polynomial term behave reasonably under Ridge.")
else:
    print("\nSome polynomial terms are still relatively large under Ridge.")

Ridge Polynomial Coefficients:
                        Features   Coefficient
0                monthly_charges  6.954305e-03
1                  total_charges -7.362245e-05
2              monthly_charges^2 -3.009334e-05
3  monthly_charges total_charges  8.604961e-07
4                total_charges^2 -6.840546e-09

Largest absolute Ridge coefficient: 0.006954305321773265

The extra polynomial term behave reasonably under Ridge.


TASK 04  
Turn the date column into usable features  
Take signup_date and make it something a model can actually learn from.

In [20]:
# parse signup_date into datetime
df['signup_date'] = pd.to_datetime(df['signup_date'])

# confirm we can read data components
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
df['signup_dayofweek'] = df['signup_date'].dt.dayofweek
df['signup_quarter'] = df['signup_date'].dt.quarter

# Days since the latest signup date
df['days_since'] = (df['signup_date'].max() - df['signup_date']).dt.days

# Check the new features
print(df[['signup_date', 'signup_year', 'signup_month', 'signup_dayofweek',
         'signup_quarter', 'days_since']].head())

# Drop the raw date column
df = df.drop(columns=['signup_date'])

  signup_date  signup_year  signup_month  signup_dayofweek  signup_quarter  \
0  2022-08-03         2022             8                 2               3   
1  2022-08-08         2022             8                 0               3   
2  2023-03-17         2023             3                 4               1   
3  2021-10-12         2021            10                 1               4   
4  2023-06-09         2023             6                 4               2   

   days_since  
0         515  
1         510  
2         289  
3         810  
4         205  


In [23]:
# Add date features to your existing feature set
date_features = [
    'signup_year',
    'signup_month',
    'signup_dayofweek',
    'signup_quarter',
    'days_since'
]
# Add date features to X
X = df.drop(columns=['churn'])

# Train / Test split again after feature engineering
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# compare model before and after adding the date features
from sklearn.pipeline import Pipeline

# Without date features
X_train_without_date = X_train.drop(columns=date_features)

num_cols = X_train_without_date.select_dtypes( include=['int64', 'float64'] ).columns.tolist()
num_cols = [ col for col in num_cols if col != 'customer_id' ]
cat_cols = X_train_without_date.select_dtypes( include=['object'] ).columns.tolist()

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_train_without_date = ohe.fit_transform(X_train_without_date[cat_cols])
model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

cv_without_date = cross_val_score(
    model, X_train_without_date, y_train, cv=5, scoring='accuracy'
).mean()

# With date features
num_cols_with_date = num_cols + cat_cols + date_features


X_train_with_date = X_train[num_cols_with_date]

X_train_with_date = ohe.fit_transform(X_train_with_date)

cv_with_date = cross_val_score(
    model, X_train_with_date, y_train, cv=5, scoring='accuracy'
).mean()

print("CV Score without date features:", cv_without_date)
print("CV Score with date features:", cv_with_date)
print("Improvement:", cv_with_date - cv_without_date)

/tmp/ipykernel_30071/3833844610.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_without_date.select_dtypes( include=['object'] ).columns.tolist()


CV Score without date features: 0.7706249999999999
CV Score with date features: 0.76625
Improvement: -0.004374999999999907


Part C — custom transformers, leakage, and the deliverable pipeline  
TASK 05  
Spot the leakage in this code, then fix it

In [33]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# # Provided Buggy code
# scaler = StandardScaler()
# X_all_scaled = scaler.fit_transform(X) # Leak
# X_train_s, X_test_s = train_test_split(X_all_scaled, random_state = 42)

# Leaky Version
scaler = StandardScaler()
# Leak
# Standard Scaler sees complete dataset before cross verificaction
X_num = X[num_col].fillna(0)
X_scaled = scaler.fit_transform(X_num)

leaky_model = LogisticRegression(max_iter=1000)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

leaky_scores = cross_val_score(
    leaky_model, X_scaled, y, cv=cv,
    scoring='accuracy'
)

print("Leaky Version")
print("-"*14)
print("CV Scores:", leaky_scores)
print("Mean CV accuracy:", leaky_scores.mean())

Leaky Version
--------------
CV Scores: [0.77   0.7675 0.76   0.765  0.7725]
Mean CV accuracy: 0.7670000000000001


* Consequence -->  
In a real deployment, when the model is being trained, it would not have access to the future/test observations. But in the leaky version, StandardScaler uses the entire dataset to calculate the mean and standard deviation before the train/test split. Therefore, the training data is transformed using information about the distribution of data that should still be unseen. This makes the evaluation slightly optimistic because the preprocessing has indirectly learned from the test set. The model itself may never directly see the test labels, but its inputs have already been influenced by information from the test rows. The preprocessing step has information about the validation data distribution that would not be available when training a real model. This can make the CV score slightly optimistic.

In [41]:
# Fixed Version
# First split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

X[num_col] = X[num_col].fillna(0)

# Preprocessor
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_col),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_col)
])

# pipeline
fixed_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# CV Scores
fixed_scores = cross_val_score(
    fixed_pipeline, X, y, cv=cv,
    scoring='accuracy'
)

print("Fixed Version")
print("-"*14)
print("CV Scores:", fixed_scores)
print("Mean CV Accuracy:", fixed_scores.mean())

Fixed Version
--------------
CV Scores: [0.765  0.775  0.775  0.765  0.7825]
Mean CV Accuracy: 0.7725


* Fix: Put preprocessing inside a Pipeline/ColumnTransformer. This causes the scaler and encoder to be fitted separately on each training fold and then applied to that fold's validation data.

In [42]:
# compare leaky vs fixed 
comparison = pd.DataFrame({
    "Version": ["Leaky", "Fixed"],
    "Mean CV Accuracy" : [leaky_scores.mean(), 
                         fixed_scores.mean()]
})

print(comparison)

  Version  Mean CV Accuracy
0   Leaky            0.7670
1   Fixed            0.7725


In [44]:
# Calculate the gap
gap = leaky_scores.mean() - fixed_scores.mean()

print("Gap between leaky and fixed:", gap)

Gap between leaky and fixed: -0.005499999999999838


* Proof -->  
Compare leaky_scores.mean() with fixed_scores.mean(). The difference is the leakage gap. If the leaky score is higher, the leaky preprocessing produced an optimistic estimate; if they are almost identical, the leakage had little numerical effect on this particular dataset, but the fixed Pipeline is still the correct approach.

TASK 06  
Assemble the full ColumnTransformer + Pipeline

In [45]:
df.isnull().sum()

customer_id          0
tenure_months        0
age                  0
monthly_charges      0
total_charges       40
contract_type        0
payment_method       0
region               0
churn                0
signup_year          0
signup_month         0
signup_dayofweek     0
signup_quarter       0
days_since           0
dtype: int64

In [48]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# load data
df = pd.read_csv("telecom_churn.csv")

# Covert signup_date to datetime
df["signup_date"] = pd.to_datetime(df['signup_date'])


# Separate X and Y
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

# Define Column Groups
num_col = ['monthly_charges', "total_charges"]
cat_col = ['contract_type', 'payment_method', 'region']
date_col = ['signup_date']


# Date Transformer
def extract_date_features(X):
    X = pd.DataFrame(X)
    date = pd.to_datetime(X.iloc[:,0])
    return np.column_stack([
        date.dt.year,
        date.dt.month,
        date.dt.dayofweek,
        date.dt.quarter
    ])

date_transformer = Pipeline([
    ('date_features', FunctionTransformer(
        extract_date_features, validate = False
    )),
    ("scaler", StandardScaler())
])


# Column Transformer
prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ]), num_col),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_col),
    ("date", date_transformer, date_col)
])

# Complete Pipeline
pipe = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000))
])

print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value=0,
                                                                                 strategy='constant')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['monthly_charges',
                                                   'total_charges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
               

In [50]:
# Test / train split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=42, stratify=y
)


# Cross Validation
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

cv_scores = cross_val_score(
    pipe, X_train, y_train, cv=cv,
    scoring='accuracy'
)

print("CV Scores:", cv_scores)
print("Mean CV Scores:", cv_scores.mean())
print("CV Standard Deviation:", cv_scores.std())

CV Scores: [0.77295918 0.75       0.77806122 0.76726343 0.77749361]
Mean CV Scores: 0.7691554882822694
CV Standard Deviation: 0.010332638309907051


In [51]:
# Fit whole pipeline
pipe.fit(X_train, y_train)

# test score
test_score = pipe.score(X_test, y_test)
print("Test Accuracy:", test_score)

Test Accuracy: 0.7857142857142857


TASK 07  
Write your own custom transformer and plug it in

In [53]:
from sklearn.base import BaseEstimator, TransformerMixin

class OutlierCap(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_caps_ = X.quantile(self.lower)
        self.upper_caps_ = X.quantile(self.upper)

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        X = X.clip(lower = self.lower_caps_,
        upper = self.upper_caps_,
        axis=1)

        return X

In [58]:
# Numeric Pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    # our custom transformer
    ("outlier_cap", OutlierCap()),
    ("scaler", StandardScaler())
])

# Categorical Pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown='ignore'))
])

# column transformer
preprocessor = ColumnTransformer([
    ("numeric", num_pipe, num_col),
    ("categorical", cat_pipe, cat_col)
])

# Complete Pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('outlier_cap',
                                                                   OutlierCap()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['monthly_charges',
                                                   'total_charges']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
     

In [60]:
# Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model, X, y, cv=cv, scoring='accuracy'
)

# Print results
print("Cross validation Score:")
print(scores)

print("\nMean CV accuracy:")
print(scores.mean())

Cross validation Score:
[0.7825 0.7725 0.785  0.7675 0.775 ]

Mean CV accuracy:
0.7765


In [62]:
# Confirm no data leakage
# Create one training / validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit the compelete pipeline only on training data
model.fit(X_train, y_train)

# Get the fitted custom transformer
outlier_cap = (
    model
    .named_steps['preprocessor']
    .named_transformers_["numeric"]
    .named_steps["outlier_cap"]
)

# Display the caps learned from training data
print("Caps learned from Training Data:")
print("\nLower Caps:")
print(outlier_cap.lower_caps_)
print("Uppeer Caps:") 
print(outlier_cap.upper_caps_)

Caps learned from Training Data:

Lower Caps:
0    18.0000
1    33.3173
Name: 0.01, dtype: float64
Uppeer Caps:
0     124.9325
1    6574.5043
Name: 0.99, dtype: float64


No data leakage occurs because OutlierCap is inside the sklearn Pipeline. During each cross-validation fold, fit() learns the percentile-based caps only from that fold's training data. The validation fold is only passed through transform() using the caps learned from the training fold.